In [15]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

In [16]:
X = np.load("../processed/X_no_norm.npy")
y = np.load("../processed/y_no_norm.npy")
subjects = np.load("../processed/subjects_no_norm.npy")

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [17]:
print(len(np.unique(subjects)))
print(np.unique(subjects))

103
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  44  45  46  47  48  49  50  51  52  53  54  55
  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73
  74  75  76  77  78  79  80  81  82  83  84  85  86  87  90  91  93  94
  95  96  97  98  99 101 102 103 105 106 107 108 109]


In [18]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [19]:
def build_model():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256
    )(x)

    x = Dropout(0.2)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [20]:
# Fold 1 only

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[
    train_sub_idx
]

test_subjects = unique_subjects[
    test_sub_idx
]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:")
print(X_train.shape)

print("\nTest Shape:")
print(X_test.shape)

print("\nTrain Labels:")
print(
    np.unique(
        y_train,
        return_counts=True
    )
)

print("\nTest Labels:")
print(
    np.unique(
        y_test,
        return_counts=True
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape:
(4140, 7, 297)

Test Shape:
(495, 7, 297)

Train Labels:
(array([0, 1]), array([2083, 2057]))

Test Labels:
(array([0, 1]), array([254, 241]))


In [22]:
scaler = StandardScaler()

X_train_flat = X_train.reshape(-1,297)
X_test_flat = X_test.reshape(-1,297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

In [23]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model = build_model()

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - accuracy: 0.6836 - loss: 0.5677 - val_accuracy: 0.7874 - val_loss: 0.4533
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.7815 - loss: 0.4454 - val_accuracy: 0.7657 - val_loss: 0.4810
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.8196 - loss: 0.3734 - val_accuracy: 0.7754 - val_loss: 0.4735
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.8658 - loss: 0.3044 - val_accuracy: 0.7850 - val_loss: 0.5033
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.8916 - loss: 0.2461 - val_accuracy: 0.7923 - val_loss: 0.5879
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.9165 - loss: 0.1945 - val_accuracy: 0.7874 - val_loss: 0.6461


In [24]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step
              precision    recall  f1-score   support

           0       0.72      0.85      0.78       254
           1       0.81      0.66      0.72       241

    accuracy                           0.76       495
   macro avg       0.76      0.75      0.75       495
weighted avg       0.76      0.76      0.75       495



In [25]:
acc_list = []
prec_list = []
rec_list = []

fold = 1

for train_sub_idx, test_sub_idx in subject_kfold.split(
    unique_subjects
):

    print(f"\n========== Fold {fold} ==========")

    train_subjects = unique_subjects[
        train_sub_idx
    ]

    test_subjects = unique_subjects[
        test_sub_idx
    ]

    train_mask = np.isin(
        subjects,
        train_subjects
    )

    test_mask = np.isin(
        subjects,
        test_subjects
    )

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    print("Train:", X_train.shape)
    print("Test :", X_test.shape)

    # -----------------------------
    # Standardization
    # -----------------------------
    scaler = StandardScaler()

    X_train_flat = X_train.reshape(
        -1,
        297
    )

    X_test_flat = X_test.reshape(
        -1,
        297
    )

    X_train_flat = scaler.fit_transform(
        X_train_flat
    )

    X_test_flat = scaler.transform(
        X_test_flat
    )

    X_train = X_train_flat.reshape(
        X_train.shape
    )

    X_test = X_test_flat.reshape(
        X_test.shape
    )

    # -----------------------------
    # Model
    # -----------------------------
    model = build_model()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    # -----------------------------
    # Prediction
    # -----------------------------
    pred = model.predict(
        X_test,
        verbose=0
    )

    pred = (
        pred > 0.5
    ).astype(int)

    acc = accuracy_score(
        y_test,
        pred
    )

    prec = precision_score(
        y_test,
        pred
    )

    rec = recall_score(
        y_test,
        pred
    )

    print(
        f"Accuracy : {acc:.4f}"
    )

    print(
        f"Precision: {prec:.4f}"
    )

    print(
        f"Recall   : {rec:.4f}"
    )

    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)

    fold += 1


========== Fold 1 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7636
Precision: 0.7095
Recall   : 0.8714

========== Fold 2 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7495
Precision: 0.7368
Recall   : 0.7840

========== Fold 3 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.8020
Precision: 0.8072
Recall   : 0.8008

========== Fold 4 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7689
Precision: 0.7799
Recall   : 0.7376

========== Fold 5 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7089
Precision: 0.7320
Recall   : 0.6425

========== Fold 6 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7378
Precision: 0.7901
Recall   : 0.6413

========== Fold 7 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7756
Precision: 0.7470
Recall   : 0.8304

========== Fold 8 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7311
Precision: 0.7452
Re

In [26]:
print("\n========== FINAL ==========")

print(
    "Mean Accuracy:",
    np.mean(acc_list)
)

print(
    "Mean Precision:",
    np.mean(prec_list)
)

print(
    "Mean Recall:",
    np.mean(rec_list)
)

print()

print(
    "Accuracy SD:",
    np.std(acc_list)
)

print(
    "Precision SD:",
    np.std(prec_list)
)

print(
    "Recall SD:",
    np.std(rec_list)
)


========== FINAL ==========
Mean Accuracy: 0.7524040404040404
Mean Precision: 0.7481364197412368
Mean Recall: 0.7588757286289305

Accuracy SD: 0.025158645697276494
Precision SD: 0.0318006418068736
Recall SD: 0.07465082487340823
